# Text HMTL V3 训练 (Colab)

**目标**: 优化4类分类准确率，特别是平静类和激活消极类

**改进**:
- 扩充训练数据: 319 → 420 条
- 平静类: 27 → 76 条
- 激活消极类: 109 → 158 条
- 加强4类分类权重

## 1. 环境检查

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

!pip install transformers -q

## 2. 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ========== 修改这里的路径 ==========
DRIVE_PATH = "/content/drive/MyDrive"
# ===================================

TRAIN_DATA = f"{DRIVE_PATH}/training_set_hmtl_expanded.json"
MODEL_SAVE_PATH = f"{DRIVE_PATH}/text_hmtl_v3_best.pt"

print(f"训练数据存在: {os.path.exists(TRAIN_DATA)}")

## 3. 定义模型

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel


class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance"""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        p_t = torch.exp(-ce_loss)
        focal_weight = (1 - p_t) ** self.gamma
        
        if self.alpha is not None:
            if isinstance(self.alpha, torch.Tensor):
                alpha_t = self.alpha[targets]
                focal_weight = alpha_t * focal_weight
        
        focal_loss = focal_weight * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss


class TextHMTLModelV3(nn.Module):
    """
    Text HMTL V3 - 优化4类分类
    
    输出:
    - label_4_logits: 主力输出，用于多模态融合
    - label_3_logits: 辅助
    - label_7_logits: 添头，保留但不调用
    - arousal, valence: 连续值
    """
    
    def __init__(self, bert_model_name='bert-base-chinese', dropout=0.3):
        super().__init__()
        
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden_size = self.bert.config.hidden_size  # 768
        
        # 4类分类头 (主力)
        self.classifier_4 = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout * 0.67),
            nn.Linear(256, 4)
        )
        
        # 3类分类头
        self.classifier_3 = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout * 0.67),
            nn.Linear(128, 3)
        )
        
        # 7类分类头 (添头)
        self.classifier_7 = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout * 0.67),
            nn.Linear(256, 7)
        )
        
        # Arousal 回归头
        self.arousal_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        
        # Valence 回归头
        self.valence_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Tanh()
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        
        return {
            'label_4_logits': self.classifier_4(cls_output),
            'label_3_logits': self.classifier_3(cls_output),
            'label_7_logits': self.classifier_7(cls_output),
            'arousal': self.arousal_head(cls_output).squeeze(-1),
            'valence': self.valence_head(cls_output).squeeze(-1)
        }


print("模型定义完成!")

## 4. 定义数据集和损失函数

In [ ]:
import json
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer


class TextEmotionDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=128):
        with open(data_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        self.tokenizer = tokenizer
        self.max_length = max_length
        print(f"加载数据: {len(self.data)} 条")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        encoding = self.tokenizer(
            item['text'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # 处理 label_7（可能不存在）
        label_7 = item.get('label_7', 6)  # 默认平静
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label_4': torch.tensor(item['label_4'], dtype=torch.long),
            'label_3': torch.tensor(item['label_3'], dtype=torch.long),
            'label_7': torch.tensor(label_7, dtype=torch.long),
            'arousal': torch.tensor(item['arousal'], dtype=torch.float),
            'valence': torch.tensor(item['valence'], dtype=torch.float),
        }


class HMTLLossV3(nn.Module):
    """
    HMTL Loss V3 - 强调4类分类
    """
    def __init__(self, class_weights_4=None, class_weights_7=None):
        super().__init__()
        
        # 4类使用 Focal Loss (主力)
        self.loss_4 = FocalLoss(alpha=class_weights_4, gamma=2.0)
        
        # 3类和7类使用普通交叉熵
        self.loss_3 = nn.CrossEntropyLoss()
        self.loss_7 = nn.CrossEntropyLoss(weight=class_weights_7)
        
        # 回归损失
        self.mse_loss = nn.MSELoss()
    
    def forward(self, outputs, targets):
        # 4类损失 (权重最高)
        l4 = self.loss_4(outputs['label_4_logits'], targets['label_4'])
        
        # 3类损失
        l3 = self.loss_3(outputs['label_3_logits'], targets['label_3'])
        
        # 7类损失 (添头，权重低)
        l7 = self.loss_7(outputs['label_7_logits'], targets['label_7'])
        
        # A/V 损失
        l_a = self.mse_loss(outputs['arousal'], targets['arousal'])
        l_v = self.mse_loss(outputs['valence'], targets['valence'])
        
        # 总损失: 4类权重最高
        total = 1.5 * l4 + 0.8 * l3 + 0.5 * l7 + 0.8 * l_a + 0.5 * l_v
        
        return {
            'total_loss': total,
            'loss_4': l4.item(),
            'loss_3': l3.item(),
            'loss_7': l7.item(),
        }


print("数据集和损失函数定义完成!")

## 5. 加载数据

In [ ]:
from sklearn.model_selection import train_test_split

# 加载分词器
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')

# 加载数据
with open(TRAIN_DATA, 'r', encoding='utf-8') as f:
    all_data = json.load(f)

print(f"总样本数: {len(all_data)}")

# 统计4类分布
label_4_counts = {0: 0, 1: 0, 2: 0, 3: 0}
for item in all_data:
    label_4_counts[item['label_4']] += 1

LABEL_4_NAMES = {0: '积极', 1: '激活消极', 2: '非激活消极', 3: '平静'}
print("\n4类分布:")
for label_id, name in LABEL_4_NAMES.items():
    count = label_4_counts[label_id]
    print(f"  [{label_id}] {name}: {count} ({count/len(all_data)*100:.1f}%)")

# 划分训练集和验证集
train_data, val_data = train_test_split(all_data, test_size=0.15, random_state=42)

# 保存临时文件
with open('/content/train_temp.json', 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False)
with open('/content/val_temp.json', 'w', encoding='utf-8') as f:
    json.dump(val_data, f, ensure_ascii=False)

# 创建数据集
train_dataset = TextEmotionDataset('/content/train_temp.json', tokenizer)
val_dataset = TextEmotionDataset('/content/val_temp.json', tokenizer)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n训练集: {len(train_dataset)}, 验证集: {len(val_dataset)}")
print(f"训练批次: {len(train_loader)}, 验证批次: {len(val_loader)}")

## 6. 创建模型和优化器

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 创建模型
model = TextHMTLModelV3().to(device)

# 4类权重 (平静和激活消极权重更高)
total = sum(label_4_counts.values())
CLASS_WEIGHTS_4 = torch.tensor([
    total / (4 * label_4_counts[0]),  # 积极
    total / (4 * label_4_counts[1]) * 1.2,  # 激活消极 (加权)
    total / (4 * label_4_counts[2]),  # 非激活消极
    total / (4 * label_4_counts[3]) * 1.5,  # 平静 (加权更多)
]).to(device)

print(f"4类权重: {CLASS_WEIGHTS_4}")

# 损失函数
criterion = HMTLLossV3(class_weights_4=CLASS_WEIGHTS_4)

# 分层学习率
LEARNING_RATE = 2e-5

bert_params = list(model.bert.parameters())
head_params = (
    list(model.classifier_4.parameters()) +
    list(model.classifier_3.parameters()) +
    list(model.classifier_7.parameters()) +
    list(model.arousal_head.parameters()) +
    list(model.valence_head.parameters())
)

optimizer = torch.optim.AdamW([
    {'params': bert_params, 'lr': LEARNING_RATE},
    {'params': head_params, 'lr': LEARNING_RATE * 5}
])

# 学习率调度
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

print("模型和优化器创建完成!")

## 7. 训练

In [ ]:
from tqdm import tqdm

def evaluate(model, dataloader, device):
    model.eval()
    correct_4 = 0
    correct_3 = 0
    correct_7 = 0
    total = 0
    
    # 每类统计
    class_correct = {0: 0, 1: 0, 2: 0, 3: 0}
    class_total = {0: 0, 1: 0, 2: 0, 3: 0}
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            targets = {k: v.to(device) for k, v in batch.items() 
                      if k not in ['input_ids', 'attention_mask']}
            
            outputs = model(input_ids, attention_mask)
            
            pred_4 = outputs['label_4_logits'].argmax(dim=1)
            pred_3 = outputs['label_3_logits'].argmax(dim=1)
            pred_7 = outputs['label_7_logits'].argmax(dim=1)
            
            correct_4 += (pred_4 == targets['label_4']).sum().item()
            correct_3 += (pred_3 == targets['label_3']).sum().item()
            correct_7 += (pred_7 == targets['label_7']).sum().item()
            total += targets['label_4'].size(0)
            
            # 每类统计
            for i in range(len(pred_4)):
                true_label = targets['label_4'][i].item()
                class_total[true_label] += 1
                if pred_4[i].item() == true_label:
                    class_correct[true_label] += 1
    
    return {
        'acc_4': correct_4 / total if total > 0 else 0,
        'acc_3': correct_3 / total if total > 0 else 0,
        'acc_7': correct_7 / total if total > 0 else 0,
        'class_acc': {k: class_correct[k]/class_total[k] if class_total[k] > 0 else 0 
                      for k in range(4)}
    }


NUM_EPOCHS = 15
best_acc_4 = 0

print("="*60)
print("开始训练 Text HMTL V3")
print(f"Epochs: {NUM_EPOCHS}, Batch Size: {BATCH_SIZE}")
print("目标: 优化4类准确率，特别是平静和激活消极")
print("="*60)

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = {k: v.to(device) for k, v in batch.items() 
                  if k not in ['input_ids', 'attention_mask']}
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss_dict = criterion(outputs, targets)
        
        loss_dict['total_loss'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss_dict['total_loss'].item()
        pbar.set_postfix({'loss': f"{loss_dict['total_loss'].item():.4f}"})
    
    scheduler.step()
    
    # 评估
    eval_result = evaluate(model, val_loader, device)
    
    print(f"\nEpoch {epoch+1}: Loss={epoch_loss/len(train_loader):.4f}")
    print(f"  4类: {eval_result['acc_4']*100:.2f}%, 3类: {eval_result['acc_3']*100:.2f}%, 7类: {eval_result['acc_7']*100:.2f}%")
    print(f"  每类4类准确率:")
    for label_id, name in LABEL_4_NAMES.items():
        print(f"    {name}: {eval_result['class_acc'][label_id]*100:.1f}%")
    
    # 保存最佳模型
    if eval_result['acc_4'] > best_acc_4:
        best_acc_4 = eval_result['acc_4']
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'epoch': epoch,
            'acc_4': best_acc_4,
            'class_acc': eval_result['class_acc']
        }, MODEL_SAVE_PATH)
        print(f"  -> 保存最佳模型: 4类={best_acc_4*100:.2f}%")

print("\n" + "="*60)
print(f"训练完成! 最佳4类准确率: {best_acc_4*100:.2f}%")
print(f"模型保存到: {MODEL_SAVE_PATH}")
print("="*60)

## 8. 测试模型

In [ ]:
# 加载最佳模型
checkpoint = torch.load(MODEL_SAVE_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"加载模型: 4类准确率={checkpoint['acc_4']*100:.2f}%")
print(f"每类准确率: {checkpoint['class_acc']}")

# 测试几个样本
test_texts = [
    "太棒了！超级开心！",
    "气死我了！太差了！",
    "好担心明天的结果",
    "心里好难受...",
    "真让人失望",
    "还行吧，一般般",
    "知道了，收到",
]

print("\n测试样本:")
for text in test_texts:
    encoding = tokenizer(text, max_length=128, padding='max_length', 
                        truncation=True, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
    
    pred_4 = outputs['label_4_logits'].argmax(dim=1).item()
    pred_7 = outputs['label_7_logits'].argmax(dim=1).item()
    arousal = outputs['arousal'][0].item()
    valence = outputs['valence'][0].item()
    
    print(f"  {text}")
    print(f"    4类: {LABEL_4_NAMES[pred_4]}, A={arousal:.2f}, V={valence:.2f}")

## 9. 下载模型到本地

In [ ]:
from google.colab import files

# 下载模型
files.download(MODEL_SAVE_PATH)
print("模型下载完成!")